<a href="https://colab.research.google.com/github/Sucharitha255/Gen-AI-Practise/blob/main/Experiment7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%pip install requests ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 53.8 MB/s eta 0:00:00


In [8]:

import os
import requests
from getpass import getpass
from ddgs import DDGS

# ==========================================
# AI FINANCIAL ASSISTANT AGENT
# ==========================================

# Enter your API key securely when prompted.
# Use a replacement key if the previous one was exposed.
API_KEY = getpass("Enter your Alpha Vantage API key: ")

# ==========================================
# TOOL 1: LOAN EMI CALCULATOR
# ==========================================

def calculate_emi(principal, annual_rate, years):

    if principal <= 0 or years <= 0 or annual_rate < 0:
        return "Error: Enter a positive loan amount and duration, and a non-negative interest rate."

    months = years * 12
    monthly_rate = annual_rate / (12 * 100)

    if monthly_rate == 0:
        emi = principal / months
    else:
        factor = (1 + monthly_rate) ** months
        emi = principal * monthly_rate * factor / (factor - 1)

    total_payment = emi * months
    total_interest = total_payment - principal

    return (
        "\n--- LOAN EMI DETAILS ---\n"
        f"Loan Amount: Rs. {principal:,.2f}\n"
        f"Annual Interest Rate: {annual_rate}%\n"
        f"Duration: {years} years\n"
        f"Monthly EMI: Rs. {emi:,.2f}\n"
        f"Total Payment: Rs. {total_payment:,.2f}\n"
        f"Total Interest: Rs. {total_interest:,.2f}\n"
    )


# ==========================================
# TOOL 2: STOCK PRICE RETRIEVAL API
# ==========================================

def get_stock_price(symbol):

    url = "https://www.alphavantage.co/query"

    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": symbol.upper(),
        "apikey": API_KEY
    }

    try:
        response = requests.get(
            url, params=params, timeout=20
        )
        response.raise_for_status()
        data = response.json()

        quote = data.get("Global Quote", {})

        if quote and quote.get("05. price"):
            return (
                "\n--- STOCK PRICE DETAILS ---\n"
                f"Symbol: {quote.get('01. symbol', symbol.upper())}\n"
                f"Price: {quote['05. price']}\n"
                f"Latest Trading Date: "
                f"{quote.get('07. latest trading day', 'Unknown')}\n"
                "Source: Alpha Vantage\n"
                "Note: Data may be delayed.\n"
            )

        message = (
            data.get("Note")
            or data.get("Information")
            or data.get("Error Message")
            or "No stock quote returned. Check the symbol and API access."
        )

        return f"Stock API message: {message}"

    except requests.RequestException as error:
        return f"Unable to retrieve stock price: {error}"


# ==========================================
# TOOL 3: FINANCIAL INFORMATION SEARCH
# ==========================================

def search_financial_info(query):

    try:
        with DDGS() as search_engine:
            results = list(
                search_engine.text(
                    query + " finance",
                    max_results=5
                )
            )

        if not results:
            return "No search results found. Try a different query."

        output = ["\n--- FINANCIAL SEARCH RESULTS ---"]

        for index, item in enumerate(results, start=1):
            output.append(
                f"\nResult {index}\n"
                f"Title: {item.get('title', 'Untitled')}\n"
                f"Summary: {item.get('body', 'No summary')}\n"
                f"Link: {item.get('href', '')}"
            )

        return "\n".join(output)

    except Exception as error:
        return f"Financial search failed: {error}"


# ==========================================
# TOOL 4: AGENT / TOOL SELECTION
# ==========================================

def financial_agent():

    print("=" * 45)
    print("       AI FINANCIAL ASSISTANT")
    print("=" * 45)

    while True:

        print("\nChoose an option:")
        print("1. Calculate Loan EMI")
        print("2. Get Stock Price")
        print("3. Search Financial Information")
        print("4. Exit")

        choice = input("\nEnter choice (1-4): ").strip()

        # Option 1: EMI calculator
        if choice == "1":

            try:
                amount = float(input("Enter loan amount (Rs.): "))
                rate = float(input("Enter annual interest rate (%): "))
                years = int(input("Enter duration (years): "))

                print(calculate_emi(amount, rate, years))

            except ValueError:
                print("Invalid input. Enter numeric values.")

        # Option 2: Stock price API
        elif choice == "2":

            symbol = input(
                "Enter stock symbol (e.g., IBM, AAPL): "
            ).strip()

            if symbol:
                print(get_stock_price(symbol))
            else:
                print("Please enter a stock symbol.")

        # Option 3: Financial search
        elif choice == "3":

            query = input(
                "Enter your financial question: "
            ).strip()

            if query:
                print(search_financial_info(query))
            else:
                print("Please enter a search query.")

        # Option 4: Exit
        elif choice == "4":
            print("Thank you for using the Financial Assistant!")
            break

        else:
            print("Invalid choice. Please select 1, 2, 3, or 4.")


# ==========================================
# START THE AGENT
# ==========================================

financial_agent()


Enter your Alpha Vantage API key: ··········
       AI FINANCIAL ASSISTANT

Choose an option:
1. Calculate Loan EMI
2. Get Stock Price
3. Search Financial Information
4. Exit

Enter choice (1-4): 1
Enter loan amount (Rs.): 100000
Enter annual interest rate (%): 2
Enter duration (years): 2

--- LOAN EMI DETAILS ---
Loan Amount: Rs. 100,000.00
Annual Interest Rate: 2.0%
Duration: 2 years
Monthly EMI: Rs. 4,254.03
Total Payment: Rs. 102,096.63
Total Interest: Rs. 2,096.63


Choose an option:
1. Calculate Loan EMI
2. Get Stock Price
3. Search Financial Information
4. Exit

Enter choice (1-4): 2
Enter stock symbol (e.g., IBM, AAPL): IBM

--- STOCK PRICE DETAILS ---
Symbol: IBM
Price: 251.4600
Latest Trading Date: 2026-10-21
Source: Alpha Vantage
Note: Data may be delayed.


Choose an option:
1. Calculate Loan EMI
2. Get Stock Price
3. Search Financial Information
4. Exit

Enter choice (1-4): 2
Enter stock symbol (e.g., IBM, AAPL): Google
Stock API message: No stock quote returned. Check the